# restore of notebook with 10.4 score

Started off as a small number of tweaks to last years winning ARChitects notebook.  I was in the process of refactoring the code out into a github repo towards a rewrite for new ideas.   

Changes:

    - a master/worker client-server was introduced, and ARC tasks were farmed out to worker one by one from the master.  This would do TTT and inference on a task per task basis, across the L4 GPUs.  I was doing inference on the LoRa adapters, so it was a bit of a slow down over the ARCitects train and merge approach.
    - Some minor hyperparameter tuning
      10 train epochs
      1 inference epoch
      lora rank=32
      alpha=32
      min_prob=0.17
      max_seq_length_train = 6144
    - added a new collator which skipped examples during training (not sure that was even working at this point, was just playing around)
    - DFS was tweaked (was also just playing around).  added the ability to skip 1-2 bad probabilties - without ruining the score - in particular I was hoping it would work on ambigous tasks.  Extended the amount of probability if the grid was longer.

The jump from adding the collator and tweaked DFS was from 8 to 10.4.  I don't know if things would improved if I stuck with this approach.  I really wanted to move onto ideas which involved training my own models.  Unfortnately this was about mid May, and it was ultimately the end of the road for this approach.  I did a more wholesale rewrite of the code to aid new ideas and unfortnately the scores all dwindled to about 6-7%... so I obviously broke something.  My ideas didn't get much traction and I only got back to this in the final few weeks of the competition after a 3-4 month break.

I have restored the code this notebook as it was my best score I achieved.  Thanks to ARCitects for providing the notebook and model intact, Runpod and Lambda Labs for providing some compute and the organisers for such a fun competition.



In [ ]:
import time

global_start_time = time.time()

# check path
import sys
sys.path.append('/kaggle/input/code-restored')
print(sys.path)

def check_data():
    'determine whether this is real run or not'
    import json
    test_path = '/kaggle/input/arc-prize-2025/arc-agi_test_challenges.json'
    f = open(test_path)
    data = json.load(f)
    return "007bbfb7" in data

test_run = check_data()
print(f"This is a test_run: {test_run}")


In [ ]:
from arc25.tects import config
conf = config.get()

import os
os.environ["WANDB_DISABLED"] = "true"

if not os.path.exists(os.path.join(conf.tmp_dir, 'unsloth_installed')):  # unsloth offline install - https://stackoverflow.com/a/51646354
    !pip uninstall --yes torch accelerate
    !pip install --no-index --find-links=/kaggle/input/unsloth-2024-9-post4/wheelhouse unsloth
    

    #!pip uninstall --yes accelerate fastai torch torchaudio transformers
    #!pip install --no-index --find-links=/kaggle/input/unsloth-2024-10-7/wheelhouse unsloth  # do not use grad_acc_fix - trains very slow
    #!sed -i 's/if ((post_check - pre_check) >= 1).sum() > 1:/if False:/g' /opt/conda/lib/python3.10/site-packages/unsloth/models/llama.py
    # fix delay bug in get_statistics()
    !sed -i 's/^def get_statistics():/def get_statistics():\n if False:/g' /opt/conda/lib/python3.10/site-packages/unsloth/models/_utils.py
    # fix faulty unsloth multi-gpu detection
    !sed -i "s/raise RuntimeError('Unsloth currently does not support multi GPU setups - but we are working on it!')/pass/g" /opt/conda/lib/python3.10/site-packages/unsloth/tokenizer_utils.py /opt/conda/lib/python3.10/site-packages/unsloth/models/llama.py /opt/conda/lib/python3.10/site-packages/unsloth/models/vision.py
    os.makedirs(os.path.join(conf.tmp_dir, 'unsloth_installed'), exist_ok=True)
    print('Unsloth installed & patched.')


In [ ]:
from arc25.tects import config

TESTING = test_run

conf = config.get()

conf.benchmark_algos = False
conf.load_replies = False
conf.use_eval_ds = False
conf.use_training_ds = False

conf.infer_params.min_prob = 0.17
conf.aug_perm_count = 10
conf.filter_task_keys = []
conf.sample_challenge_size = 0
conf.max_seq_length_train = 6144
conf.experimental_collator = True

if TESTING:
    holdback = ['22208ba4', '25e02866', '5b37cb25', '5d588b4d', '5e6bbc0b', '60d73be6',
                '68bc2e87', '6ca952ad', '8a6d367c', '8abad3cf', '981add89', '99caaf76',
                '9caf5b84', 'aaecdb9a', 'ad173014', 'b2bc3ffd', 'b5bb5719', 'c3fa4749',
                'c4d1a9ae', 'cc9053aa', 'cf5fd0ad', 'db615bd4', 'e39e9282', 'f28a3cbb',
                'fc4aaf52']

    conf.use_training_ds = True
    conf.benchmark_algos = True
    conf.load_replies = True
    conf.filter_task_keys = holdback


# XXX these are all dead with running one at a time
conf.max_inference_time_secs = 0
conf.max_train_time_secs = 0
conf.reload_model_after_training = False
conf.allow_skip_training = False
conf.n_gpus = 4

conf.max_seq_length_infer = 8192


conf.do_fast_test = False
conf.do_original_ds = True

conf.use_aug_score = True
conf.train_embedding = True
conf.aug_score_params.tp = True
conf.aug_score_params.perm = "rnd_all"
conf.perm_aug = "rnd_all"

conf.model_type = "other"
config.save("go.conf")

!cat go.conf

In [ ]:
%%writefile runner_script.py

import sys
import time
import traceback
from pprint import pprint


class QueueWriter:
    def __init__(self, queue, stream_name, gpu_id):
        self.queue = queue
        self.stream_name = stream_name
        self.gpu_id = gpu_id
        self.buffer = ""

    def write(self, text):
        self.buffer += text
        if '\n' in self.buffer:
            lines = self.buffer.split('\n')
            # Send all complete lines
            for line in lines[:-1]:
                # Avoid sending empty strings if there are multiple newlines
                if line:
                    self.queue.put(f"[{self.stream_name} {self.gpu_id}]: {line}")

            # Keep the last partial line in buffer
            self.buffer = lines[-1]

    def flush(self):
        # Send any remaining buffered text, typically called on exit
        if self.buffer:
            self.queue.put(f"[{self.stream_name} {self.gpu_id}]: {self.buffer}")
            self.buffer = ""

    def close(self, *args, **kwds):
        pass


class Runner:
    def __init__(self, task_input_queue, result_output_queue, gpu_id):
        self.task_input_queue = task_input_queue
        self.result_output_queue = result_output_queue
        self.gpu_id = gpu_id
        self.init_all()

    def init_all(self):
        from arc25.tects import config
        from arc25.tects.entry import load_dataset
        from arc25.tects.entry import init, load_model, mem_info

        # XXX get for side effects...  we can remove when we push fix for load
        config.get()    
        self.conf = config.load("go.conf")
        
        # XXX need to decide whether to remove replies here
        self.orig_dataset = load_dataset(self.conf, verbose=True).remove_replies()
        self.orig_dataset.is_orig = True
        
        # warm fuzzy seeing everyhing is ok
        if self.gpu_id == 0:
            pprint(self.conf)
            print(self.orig_dataset.keys)
            
        # set up GPU and import unsloth
        init(self.gpu_id)    
        print(f"set up GPU {self.gpu_id}")
    
        # load model and set the peft for training
        self.model, self.formatter = load_model(self.conf, self.conf.base_model_path)
        mem_info(self.gpu_id)

    def run_task(self, task_id):
        from arc25.tects.entry import augment_dataset, run_training, do_inference
        from arc25.tects.entry import set_model_peft, mem_info
        try:
            s0 = time.time()
            print(f"---> Doing task {task_id}")

            model = set_model_peft(self.conf, self.model)
            
            small_ds = self.orig_dataset.filter_keys(task_id)
            dataset = augment_dataset(self.conf, self.formatter, small_ds, train=True)

            print("---> training")
            run_training(self.conf, model, self.formatter, dataset)
            mem_info(self.gpu_id)
            s1 = time.time()
            
            print("---> inference")
            # inference!
            inference_dataset = augment_dataset(self.conf, self.formatter, small_ds, train=False)
            do_inference(self.conf, model, self.formatter, small_ds, inference_dataset)

            mem_info(self.gpu_id)
            s2 = time.time()
            
            times = s1 - s0, s2 - s1

            # Put intermediate result onto the queue
            self.result_output_queue.put(('done', task_id, times))
        
        except AssertionError as e:
            try:
                self.result_output_queue.put(('error', task_id, f"Fatal assertion error: {e}"))
            except:
                pass
            print("Assertion error: ", str(e))
            _, _, tb = sys.exc_info()
            traceback.print_exc()
    
        except Exception as e:
            try:
                self.result_output_queue.put(('error', task_id, f"Fatal exception: {e}"))
            except:
                pass
    
            print("Fatal error: ", str(e))
            _, _, tb = sys.exc_info()
            traceback.print_exc()

    def loop(self):
        while True:
            print(f"Worker locked and loaded {self.gpu_id}!")

            # Get task arguments from the input queue. This will block until a task is available.
            # The item should be None (to signal shutdown) or a tuple of task arguments.
            task_id = self.task_input_queue.get()

            # Sentinel value to stop the worker
            if task_id is None:
                print(f"Worker gpu_id: {self.gpu_id} received stop signal. Exiting.")
                break

            self.run_task(task_id)

    


def worker_main_loop(task_input_queue, result_output_queue, output_logging_queue, gpu_id):
    """
    This is the target function for each persistent worker process.
    It continuously fetches tasks from the task_input_queue, processes them
    using the provided solver functions, and puts results onto the result_output_queue.
    """

    # swap out stdout, stderr
    orig_stdout, sys.stdout = sys.stdout, QueueWriter(output_logging_queue, "stdout", gpu_id)
    orig_stderr, sys.stderr = sys.stderr, QueueWriter(output_logging_queue, "stderr", gpu_id)
    
    try:
        Runner(task_input_queue, result_output_queue, gpu_id).loop()
    finally:
        sys.stdout.flush()
        sys.stderr.flush()
        sys.stdout = orig_stdout
        sys.stderr = orig_stderr


In [ ]:
import time

from collections import deque

import multiprocessing as mp
from multiprocessing.queues import Empty

# XXX hard coded here, should be in conf
# 45 mins definitely should be long enough to finish up
TIME_TO_RUN = 3600 * 11.25

class MasterProcess:
    ''' we create 3 queues.  They are all N-N safe queues.
      1. for logging output from workers
      2. for results from workers (there isnt much but an ack currently
      3. to send request from master to workers for task_id
      '''
    def __init__(self, worker_fn, n_gpus):
        self.n_gpus = n_gpus

        self.ctx = mp.get_context('spawn')

        # For results FROM workers
        self.result_queue = self.ctx.Queue()

        # For tasks TO workers
        self.task_input_queue = self.ctx.Queue()

        # Queue for stdout/stderr from workers ---
        self.output_logging_queue = self.ctx.Queue()

        # Tracks tasks sent to workers (key: (task_id), value: {'start_time': ...})
        self.dispatched_tasks_info = {}

        # Process Management ---
        self.worker_pool = []

    def start_solver_workers(self):
        print(f"Starting {self.n_gpus} solver worker processes...")
        for gpu_id in range(self.n_gpus):
            name = f"worker_{gpu_id}"
            process = self.ctx.Process(target=worker_main_loop,
                                       args=(self.task_input_queue,
                                             self.result_queue,
                                             self.output_logging_queue,
                                             gpu_id),
                                       name=name)
            process.daemon = True
            process.start()
            self.worker_pool.append(process)

        print(f"{len(self.worker_pool)} worker processes started.")

    def stop_solver_workers(self):
        try:
            self.check_logging()
            print("Sending stop signal to all worker processes...")
            for _ in self.worker_pool:
                try:
                    self.task_input_queue.put(None)
                except Exception as e:
                    print(f"Error putting None to task_input_queue (worker may already be dead): {e}")

            time.sleep(2)
            self.check_logging()
                        
            print("Joining worker processes...")
            for i, process in enumerate(self.worker_pool):
                # Wait for the process to finish
                process.join(timeout=10)
                if process.is_alive():
                    print(f"Worker {process.name} did not terminate gracefully after 10s. Terminating...")
    
                    # Force terminate if it doesn't stop
                    process.terminate()
    
                    # Wait for termination
                    process.join()

            self.check_logging()

        except Exception as e:
            print(f"Failed during stop_solver_workers {e}")
        
        self.worker_pool = []
        print("All worker processes have been stopped and joined.")

    def run(self, keys):
        # tasks left to farm out 
        tasks_to_dispatch = deque(keys)

        # tasks sent to worker (just a statistic)
        tasks_sent_count = 0

        # count of tasks sent back from worker, and we have processed (we dont do much, just log)
        tasks_processed = 0

        # the number of tasks we need process until finished
        total_tasks_to_process_fully = len(tasks_to_dispatch)

        exit_success = True
        while True:
            if tasks_to_dispatch:
                if time.time() > global_start_time + TIME_TO_RUN:
                    tasks_failed_to_run = len(tasks_to_dispatch)
                    tasks_to_dispatch = []
                    total_tasks_to_process_fully -= tasks_failed_to_run
                    print(f"WARNING: timeout - failed to run {tasks_failed_to_run} tasks")
                    print(f"Will wait for processes to complete ({tasks_sent_count}/{tasks_processed}/{total_tasks_to_process_fully})")

            # Dispatch new tasks if there's capacity and tasks are available
            while tasks_to_dispatch and len(self.dispatched_tasks_info) < self.n_gpus:
                task_id_key = tasks_to_dispatch.popleft()

                self.task_input_queue.put(task_id_key)
                self.dispatched_tasks_info[task_id_key] = {'start_time': time.time()}
                tasks_sent_count += 1
                print(f"Dispatching task {task_id_key} ({tasks_sent_count}/{tasks_processed}/{total_tasks_to_process_fully})")

            # Process available results
            self.check_logging()
            for result_type, task_id, result_data in self.process():
                if result_type == "error":
                    #tasks_failed_to_run = len(tasks_to_dispatch)
                    print(f"ERROR for {task_id} : {result_data}")
                    print(f"WARNING: failed to run {tasks_failed_to_run} tasks")
                else:
                    assert result_type == "done"
                    print(f"Done {task_id} : {result_data}")
                tasks_processed += 1
                    
            if tasks_processed >= total_tasks_to_process_fully:
                print("All done!")
                break

            time.sleep(0.1)
            
            self.worker_pool = [p for p in self.worker_pool if p.is_alive()]
            if len(self.worker_pool) != self.n_gpus:
                print("some of the processes died :( - exiting")
                exit_success = False
                break
        return exit_success

    def check_logging(self):
        try:
            while True:
                log_message = self.output_logging_queue.get_nowait()
                print(log_message)

        except Empty:
            # No more log messages for now
            pass

        except Exception as e:
            print(f"Error in check_logging {e}")

    def process(self):
        """Processes all currently available results from the result_queue."""
        try:
            while True:
                result_type, task_id, result_data = self.result_queue.get_nowait()
                print(f"master: Rx {result_type} {task_id} {result_data}")

                if task_id in self.dispatched_tasks_info:
                    del self.dispatched_tasks_info[task_id]
                else:
                    print(f'WARNING - task_id {task_id} not in self.dispatched_tasks_info')
                    yield "error", None, "FATAL ERROR"
                    continue

                if result_type not in 'done error'.split():
                    print(f'WARNING - weird result_type {result_type}')
                    yield "error", None, "FATAL ERROR"
                    continue

                yield result_type, task_id, result_data

        except Empty:
            # No more log messages for now
            pass

        except Exception as e:
            print(f"Error processing worker output queue: {e}")


In [ ]:
from arc25.tects.entry import load_dataset
from runner_script import worker_main_loop

def get_keys_ordered(ds):
    from arc25.tects.arc_loader import arc_formatter_premix_3 as MyFormatter

    # multi replies doesnt really matter
    # ds = ds.split_multi_replies()
    formatter = MyFormatter()
    class Tokenizer:
        pass
    formatter.tokenizer = Tokenizer()
    formatter.tokenizer.eos_token = "."

    sort_me = []
    for k in ds.keys:
        f0 = ds.get(k, formatter=formatter)["input"]
        f1 = ds.get(k, formatter=formatter)["query"]
        f2 = ds.get(k, formatter=formatter)["text"]
        sort_me.append((k, len(f0) + len(f1), len(f2)))
    sort_me.sort(key=lambda x: x[1])
    for k, l0, l1 in sort_me:
        print(k, l0, l1)
        yield k

# we don't need replies, just the keys here
test_set = load_dataset(conf, verbose=True).remove_replies()

# sort keys from shortest to longest (for input)
# going to assume longer then length, longer they will take to run
keys = list(get_keys_ordered(test_set))

master = MasterProcess(worker_main_loop, conf.n_gpus)
master.start_solver_workers()
result = master.run(keys)
master.stop_solver_workers()
if not result:
    raise Exception("not good")


In [ ]:
from arc25.tects.final_pass import scoring_agg

scoring_agg(conf)


In [ ]:
# remove all this (was for an ensemble)
import json
from arc25.common.grid import Grid
from arc25.common import submit

DUMMY = [[0, 0], [0, 0]]
DUMMY2 = [[0]]


def get_indices_and_attempts(s):
    for k, v in s.data.items():
        assert isinstance(v, list)
        for i, attempts in enumerate(v):
            assert(len(attempts) == 2)
            first, second = attempts['attempt_1'], attempts['attempt_2']
            if first == DUMMY:
                assert second == DUMMY
            yield k, i, [x for x in [first, second] if x != DUMMY]


def merge(submission):
    llm_data = json.loads(open("/kaggle/working/submission_llm.json").read())

    # merge in first?
    for k, i, attempts in list(get_indices_and_attempts(submission)):
        # check llm_data is set
        try:
            llm_attempts = llm_data[k][i]
    
        except KeyError:
            continue
    
        try:
            first, _ = llm_attempts['attempt_1'], llm_attempts['attempt_2']
        except Exception as exc:
            print(f"json corrupt {exc} {llm_attempts}")
            continue
    
        if first != DUMMY and first != DUMMY2 and first not in attempts:
            attempt_number = len(attempts) + 1
            if attempt_number <= 2:
                nr = i + 1
                submission.attempt(k, attempt_number, Grid(first), nr)
                print("ADDING LLM FIRST IN")
    
    # merge in second?
    for k, i, attempts in list(get_indices_and_attempts(submission)):
        # check llm_data is set
        try:
            llm_attempts = llm_data[k][i]
    
        except KeyError:
            continue
    
        try:
            _, second = llm_attempts['attempt_1'], llm_attempts['attempt_2']
        except Exception as exc:
            print(f"json corrupt {exc} {llm_attempts}")
            continue
    
        if second != DUMMY and second != DUMMY2 and second not in attempts:
            attempt_number = len(attempts) + 1
            if attempt_number <= 2:
                nr = i + 1
                submission.attempt(k, attempt_number, Grid(second), nr)
                print("ADDING LLM SECOND IN")



def main():
    submission = submit.Submission()

    print("attempting merge submissions")
    try:
        merge(submission)
    except Exception as exc:
        print(f"merging submission failed: {exc}")
    
    if not test_run:
        !rm -rf /kaggle/working/*
        
    time.sleep(2)
    submission.save()
    time.sleep(2)

    print("DONE")

    
main()
